# BPE Tokenization and Embeddings for GPT-2

This notebook builds a complete understanding of how GPT-2 converts raw text into the dense vector representations that feed into its transformer blocks. We cover three foundational components:

1. **Byte-Pair Encoding (BPE) tokenization** -- the algorithm GPT-2 uses to break text into subword tokens. We implement a BPE trainer from scratch, watch merges happen in real time, and build working `encode()` / `decode()` functions.

2. **Token embeddings** -- a learned lookup table (`nn.Embedding`) that maps each integer token ID to a dense vector.

3. **Positional embeddings** -- GPT-2 uses *learned* position embeddings (not sinusoidal). We create them, inspect their structure, and combine them with token embeddings to produce the final input representation.

Along the way we visualize token frequencies, PCA projections of the embedding space, and cosine-similarity heatmaps. An ablation section at the end shows concretely why positional embeddings matter.

All code is self-contained -- no external imports beyond `torch`, `numpy`, and `matplotlib`. We use tiny dimensions (vocab ~256-512, embed_dim=64) so every cell runs in under a second.

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
})

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")

---
## Part 1 -- Byte-Pair Encoding from Scratch

BPE starts with a base vocabulary of individual bytes (0-255) and iteratively merges the most frequent adjacent pair into a single new token. After *k* merges the vocabulary has size 256 + *k*.

GPT-2 applies BPE at the **byte level**, meaning every possible byte is already in the base vocabulary -- no unknown tokens ever arise.

In [ ]:
# Training corpus -- a small collection of sentences
corpus = [
    "The transformer architecture revolutionized natural language processing.",
    "Attention is all you need, the paper said.",
    "Language models predict the next token in a sequence.",
    "GPT-2 uses byte-pair encoding for tokenization.",
    "Embeddings map discrete tokens to continuous vector spaces.",
    "The embedding layer is the first layer of the transformer.",
    "Positional embeddings encode the order of tokens in the sequence.",
    "Training large language models requires massive compute.",
    "The attention mechanism allows the model to focus on relevant tokens.",
    "Self-attention computes pairwise relationships between all tokens.",
    "The transformer processes all tokens in parallel, not sequentially.",
    "Byte-pair encoding starts with individual bytes and merges frequent pairs.",
    "Subword tokenization balances vocabulary size and coverage.",
    "The model learns to represent words as sequences of subword tokens.",
    "Pre-training on large corpora gives the model general language understanding.",
]

print(f"Corpus: {len(corpus)} sentences")
print(f"Total characters: {sum(len(s) for s in corpus)}")
print(f"\nFirst sentence bytes: {list(corpus[0].encode('utf-8'))[:20]}...")

In [ ]:
def get_pair_counts(token_sequences):
    """Count frequency of adjacent token pairs across all sequences."""
    counts = {}
    for seq in token_sequences:
        for i in range(len(seq) - 1):
            pair = (seq[i], seq[i + 1])
            counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge_pair(token_sequences, pair, new_token):
    """Replace every occurrence of `pair` with `new_token` in all sequences."""
    merged = []
    for seq in token_sequences:
        new_seq = []
        i = 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                new_seq.append(new_token)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        merged.append(new_seq)
    return merged


# Convert corpus to byte-level token sequences
token_sequences = [list(s.encode("utf-8")) for s in corpus]

print("Helper functions defined: get_pair_counts(), merge_pair()")
print(f"\nConverted {len(token_sequences)} sentences to byte sequences.")
print(f"First sequence (first 30 bytes): {token_sequences[0][:30]}...")

In [ ]:
# Train BPE: perform 30 merges
NUM_MERGES = 30

merges = []              # list of (pair, new_token_id)
next_token_id = 256      # base vocab is bytes 0-255
token_to_bytes = {i: bytes([i]) for i in range(256)}  # for decoding

print(f"Base vocabulary size: 256 (all single bytes)")
print(f"Performing {NUM_MERGES} merges...\n")

for step in range(NUM_MERGES):
    pair_counts = get_pair_counts(token_sequences)
    if not pair_counts:
        print(f"No more pairs to merge at step {step}.")
        break

    best_pair = max(pair_counts, key=pair_counts.get)
    best_count = pair_counts[best_pair]

    merges.append((best_pair, next_token_id))
    token_to_bytes[next_token_id] = token_to_bytes[best_pair[0]] + token_to_bytes[best_pair[1]]

    pair_repr = (
        token_to_bytes[best_pair[0]].decode("utf-8", errors="replace")
        + " + "
        + token_to_bytes[best_pair[1]].decode("utf-8", errors="replace")
    )
    merged_repr = token_to_bytes[next_token_id].decode("utf-8", errors="replace")

    print(f"Merge {step+1:2d}: ({best_pair[0]:3d}, {best_pair[1]:3d}) -> {next_token_id}  "
          f"[{pair_repr:>12s} -> {merged_repr:<6s}]  count={best_count}")

    token_sequences = merge_pair(token_sequences, best_pair, next_token_id)
    next_token_id += 1

vocab_size = next_token_id
print(f"\nFinal vocabulary size: {vocab_size} (256 base + {NUM_MERGES} merges)")

Notice how BPE first merges very common byte pairs like `t + h` into `th`, `e + (space)` or `(space) + t`, and gradually builds longer subword tokens. This is exactly how GPT-2's tokenizer was trained -- except on a vastly larger corpus with ~50,000 merges.

In [ ]:
def encode(text):
    """Encode a string into a list of BPE token IDs."""
    tokens = list(text.encode("utf-8"))
    for (pair, new_id) in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(new_id)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


def decode(token_ids):
    """Decode a list of BPE token IDs back to a string."""
    byte_seq = b"".join(token_to_bytes[t] for t in token_ids)
    return byte_seq.decode("utf-8", errors="replace")


# Tokenization examples
test_strings = [
    "The transformer",
    "Attention is all you need",
    "token embeddings",
    "language model",
    "zzzzz xylophone",  # unusual text -- falls back to bytes
]

for s in test_strings:
    ids = encode(s)
    decoded = decode(ids)
    subwords = [token_to_bytes[t].decode("utf-8", errors="replace") for t in ids]
    print(f"\n  Input: {s!r}")
    print(f"  IDs:   {ids}")
    print(f"  Parts: {subwords}")
    print(f"  Round-trip: {decoded!r}  (match: {decoded == s})")

In [ ]:
# Verify round-trip fidelity on entire corpus
all_match = True
for sentence in corpus:
    ids = encode(sentence)
    reconstructed = decode(ids)
    if reconstructed != sentence:
        print(f"MISMATCH: {sentence!r} -> {reconstructed!r}")
        all_match = False

if all_match:
    print("All corpus sentences encode and decode perfectly.")

# Show compression ratio
total_bytes = sum(len(s.encode("utf-8")) for s in corpus)
total_tokens = sum(len(encode(s)) for s in corpus)
print(f"\nTotal bytes:  {total_bytes}")
print(f"Total tokens: {total_tokens}")
print(f"Compression:  {total_bytes / total_tokens:.2f} bytes per token")

Even with only 30 merges, common pairs like `th`, `the`, `in`, `on` get merged, reducing the number of tokens needed. GPT-2's full tokenizer (50,257 vocab, ~40k merges) achieves roughly 3-4 bytes per token on English text.

---
## Part 2 -- Token Embedding Lookup Table

Once we have integer token IDs, we need to convert them into dense vectors the transformer can process. This is done by a simple **lookup table** implemented as `nn.Embedding`. Each row of the table is a learnable vector for one token.

In [ ]:
EMBED_DIM = 64
MAX_SEQ_LEN = 128

# Token embedding table
token_embedding = nn.Embedding(vocab_size, EMBED_DIM)

print(f"Vocabulary size:  {vocab_size}")
print(f"Embedding dim:    {EMBED_DIM}")
print(f"Parameter count:  {vocab_size * EMBED_DIM:,}")
print(f"Weight shape:     {token_embedding.weight.shape}")

In [ ]:
# Demonstrate the lookup: token IDs -> embedding vectors
example_text = "The transformer"
example_ids = encode(example_text)
example_tensor = torch.tensor(example_ids, dtype=torch.long)

with torch.no_grad():
    embeddings = token_embedding(example_tensor)

print(f"Text:      {example_text!r}")
print(f"Token IDs: {example_ids}")
print(f"Input shape:  {example_tensor.shape}  (sequence_length,)")
print(f"Output shape: {embeddings.shape}  (sequence_length, embed_dim)")
print(f"\nFirst token (ID={example_ids[0]}) embedding (first 10 dims):")
print(f"  {embeddings[0, :10].numpy().round(4)}")

# Verify it matches direct weight lookup
direct_lookup = token_embedding.weight[example_ids[0]]
print(f"\nDirect weight[{example_ids[0]}] (first 10 dims):")
print(f"  {direct_lookup[:10].detach().numpy().round(4)}")
print(f"Match: {torch.allclose(embeddings[0], direct_lookup)}")

The embedding layer is just a matrix of shape `(vocab_size, embed_dim)`. Passing in a token ID `i` simply returns row `i` of that matrix. During training, backpropagation updates only the rows that were actually used in each batch.

---
## Part 3 -- Learned Positional Embeddings (GPT-2 Style)

Unlike the original Transformer (which uses fixed sinusoidal functions), GPT-2 uses **learned positional embeddings**. These are another `nn.Embedding` table, indexed by position (0, 1, 2, ...) rather than token ID. The model learns during training what each position "means."

In [ ]:
# Positional embedding table
pos_embedding = nn.Embedding(MAX_SEQ_LEN, EMBED_DIM)

print(f"Max sequence length: {MAX_SEQ_LEN}")
print(f"Embedding dim:       {EMBED_DIM}")
print(f"Parameter count:     {MAX_SEQ_LEN * EMBED_DIM:,}")
print(f"Weight shape:        {pos_embedding.weight.shape}")

# These are learned parameters -- initially random
print(f"\nPosition 0 embedding (first 10 dims):")
print(f"  {pos_embedding.weight[0, :10].detach().numpy().round(4)}")
print(f"Position 1 embedding (first 10 dims):")
print(f"  {pos_embedding.weight[1, :10].detach().numpy().round(4)}")

# Confirm these are learned (requires_grad=True), not fixed
print(f"\nPositional embedding parameters:")
for name, param in pos_embedding.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

---
## Part 4 -- Combined Embedding: token_emb + pos_emb

GPT-2's input representation is simply the **element-wise sum** of the token embedding and the positional embedding:

$$\mathbf{h}_i^{(0)} = \text{token\_emb}(x_i) + \text{pos\_emb}(i)$$

This combined vector is what gets fed into the first transformer block.

In [ ]:
def embed_text(text, tok_emb, pos_emb):
    """Full GPT-2-style embedding pipeline: text -> combined embeddings."""
    token_ids = encode(text)
    tokens = torch.tensor(token_ids, dtype=torch.long)
    seq_len = tokens.shape[0]
    tok_vectors = tok_emb(tokens)              # (seq_len, embed_dim)
    pos_vectors = pos_emb(torch.arange(seq_len))  # (seq_len, embed_dim)
    combined = tok_vectors + pos_vectors        # (seq_len, embed_dim)
    return combined, token_ids


# Run the full pipeline on an example
example = "Attention is all you need"

with torch.no_grad():
    combined, ids = embed_text(example, token_embedding, pos_embedding)

subwords = [token_to_bytes[t].decode("utf-8", errors="replace") for t in ids]

print(f"Text:       {example!r}")
print(f"Subwords:   {subwords}")
print(f"Token IDs:  {ids}")
print(f"Combined embedding shape: {combined.shape}")

In [ ]:
# Decompose: verify combined = token_emb + pos_emb and compare magnitudes
with torch.no_grad():
    tok_only = token_embedding(torch.tensor(ids, dtype=torch.long))
    pos_only = pos_embedding(torch.arange(len(ids)))
    manual_sum = tok_only + pos_only

print("Verifying combined = token_emb + pos_emb:")
print(f"  Max difference: {(combined - manual_sum).abs().max().item():.2e}")
print(f"  Match: {torch.allclose(combined, manual_sum)}")

tok_norms = tok_only.norm(dim=1)
pos_norms = pos_only.norm(dim=1)
combined_norms = combined.norm(dim=1)

print(f"\nVector norms (averaged across {len(ids)} tokens):")
print(f"  Token embeddings:    {tok_norms.mean().item():.4f}")
print(f"  Position embeddings: {pos_norms.mean().item():.4f}")
print(f"  Combined:            {combined_norms.mean().item():.4f}")

Both the token and positional embeddings contribute to the final representation. In a trained model, the relative magnitudes become meaningful -- the model learns to balance identity ("what token is this?") and position ("where is it in the sequence?").

---
## Part 5 -- Visualizations

### 5.1 Token Frequency Bar Chart

Let's see which tokens appear most frequently in our BPE-encoded corpus.

In [ ]:
# Count token frequencies across the encoded corpus
all_tokens = []
for sentence in corpus:
    all_tokens.extend(encode(sentence))

freq = {}
for t in all_tokens:
    freq[t] = freq.get(t, 0) + 1

top20 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:20]

labels = []
for tok_id, count in top20:
    text_repr = token_to_bytes[tok_id].decode("utf-8", errors="replace")
    display = repr(text_repr)[1:-1]
    labels.append(f"{display} ({tok_id})")

counts = [c for _, c in top20]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(labels)), counts, color="steelblue", edgecolor="white")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=55, ha="right", fontsize=9)
ax.set_ylabel("Frequency")
ax.set_title("Top 20 Most Frequent BPE Tokens in Corpus")
ax.set_xlabel("Token (text representation and ID)")

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(count), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

Common English patterns dominate: space-prefixed function words, common bigrams that got merged. The distribution follows a Zipf-like pattern, which is characteristic of natural language at any tokenization granularity.

### 5.2 Embedding PCA -- 2D Projection

We project the 64-dimensional token embeddings down to 2D using PCA to see if any structure is visible -- even in untrained (random) embeddings.

In [ ]:
# Get embeddings for all tokens that appear in the corpus
unique_tokens = sorted(freq.keys())
unique_tensor = torch.tensor(unique_tokens, dtype=torch.long)

with torch.no_grad():
    emb_matrix = token_embedding(unique_tensor)

# Center the data and compute PCA via SVD
emb_centered = emb_matrix - emb_matrix.mean(dim=0, keepdim=True)
U, S, Vt = torch.linalg.svd(emb_centered, full_matrices=False)
pca_2d = (emb_centered @ Vt[:2].T).numpy()

var_explained = (S[:2] ** 2) / (S ** 2).sum()
print(f"Variance explained by PC1: {var_explained[0]:.2%}")
print(f"Variance explained by PC2: {var_explained[1]:.2%}")

fig, ax = plt.subplots(figsize=(10, 8))

colors = ["#e74c3c" if t >= 256 else "#3498db" for t in unique_tokens]
sizes = [60 if t >= 256 else 25 for t in unique_tokens]
ax.scatter(pca_2d[:, 0], pca_2d[:, 1], c=colors, s=sizes, alpha=0.7,
           edgecolors="white", linewidths=0.5)

for i, tok_id in enumerate(unique_tokens):
    if tok_id >= 256:
        text_repr = token_to_bytes[tok_id].decode("utf-8", errors="replace")
        ax.annotate(repr(text_repr)[1:-1], (pca_2d[i, 0], pca_2d[i, 1]),
                    fontsize=7, ha="left", va="bottom",
                    xytext=(4, 4), textcoords="offset points")

ax.set_xlabel(f"PC1 ({var_explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.1%} variance)")
ax.set_title("PCA Projection of Token Embeddings (untrained)")

legend_elements = [
    mlines.Line2D([0], [0], marker="o", color="w", markerfacecolor="#3498db",
                  markersize=8, label="Single-byte tokens"),
    mlines.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e74c3c",
                  markersize=10, label="Merged (BPE) tokens"),
]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.show()

Since these embeddings are untrained (random initialization), there is no meaningful clustering yet. In a trained model, semantically related tokens (e.g., `the`, `The`, `THE`) would cluster together, and the first few principal components would capture broad linguistic dimensions.

### 5.3 Cosine Similarity Heatmap of Token Embeddings

We compute pairwise cosine similarity among the top 20 most frequent tokens to see how similar their (untrained) embeddings are.

In [ ]:
top20_ids = [tok_id for tok_id, _ in top20]
top20_tensor = torch.tensor(top20_ids, dtype=torch.long)

with torch.no_grad():
    top20_embs = token_embedding(top20_tensor)

norms = top20_embs.norm(dim=1, keepdim=True)
cos_sim = (top20_embs @ top20_embs.T) / (norms @ norms.T)
cos_sim_np = cos_sim.numpy()

tick_labels = []
for tok_id in top20_ids:
    text_repr = token_to_bytes[tok_id].decode("utf-8", errors="replace")
    tick_labels.append(repr(text_repr)[1:-1])

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cos_sim_np, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(tick_labels)))
ax.set_yticks(range(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=55, ha="right", fontsize=9)
ax.set_yticklabels(tick_labels, fontsize=9)
ax.set_title("Cosine Similarity: Top 20 Token Embeddings (untrained)")

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Cosine Similarity")

plt.tight_layout()
plt.show()

print(f"Diagonal (self-similarity): all {cos_sim_np.diagonal().mean():.4f}")
mask = ~np.eye(len(top20_ids), dtype=bool)
print(f"Off-diagonal mean: {cos_sim_np[mask].mean():.4f}")
print(f"Off-diagonal std:  {cos_sim_np[mask].std():.4f}")

With random initialization, off-diagonal cosine similarities are close to zero (as expected for random vectors in 64 dimensions). After training, this heatmap would reveal which tokens the model considers functionally similar.

---
## Part 6 -- Ablation: Why Positional Embeddings Matter

### 6.1 No Positional Embeddings

Without positional embeddings, the model has **no way to distinguish token order**. The same tokens in any arrangement produce the same set of vectors (as a bag, not a sequence).

In [ ]:
# Without positional embeddings, permuted inputs look identical
sentence_a = "the model is great"
sentence_b = "great is the model"  # same tokens, different order

ids_a = encode(sentence_a)
ids_b = encode(sentence_b)

with torch.no_grad():
    tok_a = token_embedding(torch.tensor(ids_a, dtype=torch.long))
    tok_b = token_embedding(torch.tensor(ids_b, dtype=torch.long))
    combined_a, _ = embed_text(sentence_a, token_embedding, pos_embedding)
    combined_b, _ = embed_text(sentence_b, token_embedding, pos_embedding)

bag_a_no_pos = tok_a.sum(dim=0)
bag_b_no_pos = tok_b.sum(dim=0)
bag_a_with_pos = combined_a.sum(dim=0)
bag_b_with_pos = combined_b.sum(dim=0)

cos = nn.CosineSimilarity(dim=0)

print(f"Sentence A: {sentence_a!r}")
print(f"Sentence B: {sentence_b!r}")
print(f"\n--- Without positional embeddings ---")
print(f"Sum-of-vectors cosine similarity: {cos(bag_a_no_pos, bag_b_no_pos).item():.6f}")
print(f"Sum-of-vectors L2 distance:       {(bag_a_no_pos - bag_b_no_pos).norm().item():.6f}")
print(f"\n--- With positional embeddings ---")
print(f"Sum-of-vectors cosine similarity: {cos(bag_a_with_pos, bag_b_with_pos).item():.6f}")
print(f"Sum-of-vectors L2 distance:       {(bag_a_with_pos - bag_b_with_pos).norm().item():.6f}")

In [ ]:
# Per-position breakdown: norms of token vs position vs combined embeddings
sentence = "the model is great"
ids = encode(sentence)
seq_len = len(ids)

with torch.no_grad():
    tok_vecs = token_embedding(torch.tensor(ids, dtype=torch.long))
    pos_vecs = pos_embedding(torch.arange(seq_len))
    combined_vecs = tok_vecs + pos_vecs

print(f"Sentence: {sentence!r}")
print(f"Tokens: {[token_to_bytes[t].decode('utf-8', errors='replace') for t in ids]}")
print(f"\nWithout position: embedding depends ONLY on token identity.")
print(f"With position:    embedding depends on token identity AND position.")

print(f"\n{'Position':<10} {'Token':<10} {'|tok_emb|':<12} {'|pos_emb|':<12} {'|combined|':<12}")
print("-" * 56)
for i in range(seq_len):
    tok_repr = token_to_bytes[ids[i]].decode("utf-8", errors="replace")
    print(f"{i:<10} {repr(tok_repr):<10} {tok_vecs[i].norm().item():<12.4f} "
          f"{pos_vecs[i].norm().item():<12.4f} {combined_vecs[i].norm().item():<12.4f}")

### 6.2 Random vs Learned Positional Embeddings

We compare the cosine similarity structure of (a) random positional embeddings, (b) our initialized-but-untrained learned embeddings, and (c) a smooth synthetic "trained" pattern to illustrate what training accomplishes.

In [ ]:
NUM_POS = 32  # look at first 32 positions

# (a) Fully random position embeddings
torch.manual_seed(99)
random_pos_emb = nn.Embedding(MAX_SEQ_LEN, EMBED_DIM)

# (b) Our existing learned (but untrained) embeddings -- already in pos_embedding

# (c) Synthetic "trained" pattern: nearby positions should be similar
synthetic_weight = torch.zeros(MAX_SEQ_LEN, EMBED_DIM)
for p in range(MAX_SEQ_LEN):
    for d in range(EMBED_DIM):
        synthetic_weight[p, d] = np.sin(p / (10.0 ** (d / EMBED_DIM))) + \
                                  0.1 * np.cos(p * d / 100.0)
synthetic_pos_emb = nn.Embedding(MAX_SEQ_LEN, EMBED_DIM)
with torch.no_grad():
    synthetic_pos_emb.weight.copy_(synthetic_weight)

def pos_cosine_sim(emb_layer, n):
    with torch.no_grad():
        vecs = emb_layer(torch.arange(n))
        norms = vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
        return ((vecs @ vecs.T) / (norms @ norms.T)).numpy()

sim_random = pos_cosine_sim(random_pos_emb, NUM_POS)
sim_learned = pos_cosine_sim(pos_embedding, NUM_POS)
sim_synthetic = pos_cosine_sim(synthetic_pos_emb, NUM_POS)

print(f"Computed cosine similarity matrices for {NUM_POS} positions.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

titles = ["(a) Random Embeddings", "(b) Learned (untrained)", "(c) Synthetic \"trained\""]
matrices = [sim_random, sim_learned, sim_synthetic]

for ax, title, sim_mat in zip(axes, titles, matrices):
    im = ax.imshow(sim_mat, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Position")
    ax.set_ylabel("Position")

cbar = fig.colorbar(im, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("Cosine Similarity")

fig.suptitle("Positional Embedding Similarity Patterns", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

for title, sim_mat in zip(titles, matrices):
    adj_sim = np.mean([sim_mat[i, i+1] for i in range(NUM_POS - 1)])
    far_sims = [sim_mat[i, j] for i in range(NUM_POS) for j in range(i + 10, NUM_POS)]
    far_sim = np.mean(far_sims) if far_sims else 0.0
    print(f"{title}:  adjacent_sim={adj_sim:.4f}, distant_sim={far_sim:.4f}")

The synthetic "trained" embeddings show a clear pattern: nearby positions have high cosine similarity (warm diagonal band), while distant positions are less similar. This is what a trained GPT-2 model learns -- a smooth notion of relative distance. The random and untrained embeddings lack this structure.

In [ ]:
# Similarity as a function of distance for all three schemes
def sim_by_distance(sim_mat, max_dist=None):
    """Average cosine similarity at each distance offset."""
    n = sim_mat.shape[0]
    if max_dist is None:
        max_dist = n - 1
    distances = list(range(max_dist + 1))
    avg_sims = []
    for d in distances:
        sims = [sim_mat[i, i + d] for i in range(n - d)]
        avg_sims.append(np.mean(sims))
    return distances, avg_sims


fig, ax = plt.subplots(figsize=(10, 5))

for sim_mat, label, color in [
    (sim_random, "Random", "#e74c3c"),
    (sim_learned, "Learned (untrained)", "#3498db"),
    (sim_synthetic, 'Synthetic "trained"', "#2ecc71"),
]:
    dists, sims = sim_by_distance(sim_mat)
    ax.plot(dists, sims, label=label, color=color, linewidth=2)

ax.set_xlabel("Position Distance")
ax.set_ylabel("Average Cosine Similarity")
ax.set_title("Positional Similarity Decay with Distance")
ax.legend()
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlim(0, NUM_POS - 1)

plt.tight_layout()
plt.show()

The synthetic trained embeddings show a smooth decay -- nearby positions are similar, distant ones are not. This is the kind of structure GPT-2 learns. The random/untrained curves hover around zero at all distances, providing no distance signal.

In [ ]:
# Final ablation: the same token at different positions gets different
# representations ONLY when positional embeddings are included

repeated = "the the the the the"
ids_rep = encode(repeated)
seq_len_rep = len(ids_rep)

with torch.no_grad():
    tok_vecs_rep = token_embedding(torch.tensor(ids_rep, dtype=torch.long))
    pos_vecs_rep = pos_embedding(torch.arange(seq_len_rep))
    combined_rep = tok_vecs_rep + pos_vecs_rep

def pairwise_cos(vecs):
    norms = vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
    return ((vecs @ vecs.T) / (norms @ norms.T)).numpy()

sim_no_pos = pairwise_cos(tok_vecs_rep)
sim_with_pos = pairwise_cos(combined_rep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

labels_rep = [token_to_bytes[t].decode("utf-8", errors="replace") for t in ids_rep]
tick_labels_rep = [f"{repr(l)[1:-1]}@{i}" for i, l in enumerate(labels_rep)]

im1 = ax1.imshow(sim_no_pos, cmap="RdBu_r", vmin=-1, vmax=1)
ax1.set_title("Without Positional Embeddings")
ax1.set_xticks(range(len(tick_labels_rep)))
ax1.set_yticks(range(len(tick_labels_rep)))
ax1.set_xticklabels(tick_labels_rep, rotation=45, ha="right", fontsize=7)
ax1.set_yticklabels(tick_labels_rep, fontsize=7)

im2 = ax2.imshow(sim_with_pos, cmap="RdBu_r", vmin=-1, vmax=1)
ax2.set_title("With Positional Embeddings")
ax2.set_xticks(range(len(tick_labels_rep)))
ax2.set_yticks(range(len(tick_labels_rep)))
ax2.set_xticklabels(tick_labels_rep, rotation=45, ha="right", fontsize=7)
ax2.set_yticklabels(tick_labels_rep, fontsize=7)

fig.colorbar(im2, ax=[ax1, ax2], shrink=0.8, label="Cosine Similarity")
fig.suptitle(f'Repeated tokens: "{repeated}"', fontsize=13)

plt.tight_layout()
plt.show()

print("Without positional embeddings, identical tokens have identical representations")
print("(all pairwise similarities for same token = 1.0).")
print("With positional embeddings, each occurrence is distinguishable.")

This is the core point of the ablation: without positional embeddings, a transformer treating "the the the the the" would see five **identical** input vectors. It would have no basis for attending differently to the first "the" vs. the fifth. Positional embeddings break this symmetry.

---
## Part 7 -- Full Pipeline Summary

Let's put it all together in one clean demonstration.

In [ ]:
# End-to-end: raw text -> BPE tokens -> combined embeddings
input_text = "Language models predict the next token."

token_ids = encode(input_text)
subwords = [token_to_bytes[t].decode("utf-8", errors="replace") for t in token_ids]
seq_len = len(token_ids)

with torch.no_grad():
    tok_vecs = token_embedding(torch.tensor(token_ids, dtype=torch.long))
    pos_vecs = pos_embedding(torch.arange(seq_len))
    combined = tok_vecs + pos_vecs

print("=" * 65)
print("GPT-2 Embedding Pipeline")
print("=" * 65)
print(f"\n  Raw text:       {input_text!r}")
print(f"  BPE tokens:     {subwords}")
print(f"  Token IDs:      {token_ids}")
print(f"  Sequence len:   {seq_len}")
print(f"  Embed dim:      {EMBED_DIM}")
print(f"\n  Token emb shape:    {tok_vecs.shape}")
print(f"  Position emb shape: {pos_vecs.shape}")
print(f"  Combined shape:     {combined.shape}")
print(f"\n  -> This {combined.shape} tensor is the input to the first")
print(f"     transformer block (after optional dropout).")
print("=" * 65)

In [ ]:
# Visualize the pipeline as a heatmap of the embedding matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

n_dims_show = 16  # use first 16 dimensions for visibility

data = [
    (tok_vecs[:, :n_dims_show].numpy(), "Token Embeddings"),
    (pos_vecs[:, :n_dims_show].numpy(), "Position Embeddings"),
    (combined[:, :n_dims_show].numpy(), "Combined (token + pos)"),
]

for ax, (mat, title) in zip(axes, data):
    im = ax.imshow(mat, cmap="RdBu_r", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Embedding Dimension")
    ax.set_yticks(range(seq_len))
    short_labels = [repr(s)[1:-1][:8] for s in subwords]
    ax.set_yticklabels(short_labels, fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)

axes[0].set_ylabel("Token Position")
fig.suptitle(f"Embedding Heatmaps (first {n_dims_show} dims)", fontsize=13, y=1.02)

plt.tight_layout()
plt.show()

In [ ]:
# Parameter count summary -- compare to real GPT-2
tok_params = vocab_size * EMBED_DIM
pos_params = MAX_SEQ_LEN * EMBED_DIM
total_emb_params = tok_params + pos_params

gpt2_vocab = 50257
gpt2_dim = 768
gpt2_max_pos = 1024
gpt2_tok_params = gpt2_vocab * gpt2_dim
gpt2_pos_params = gpt2_max_pos * gpt2_dim
gpt2_total_emb = gpt2_tok_params + gpt2_pos_params

print(f"{'Component':<25} {'Ours':>15} {'GPT-2 Small':>15}")
print("-" * 57)
print(f"{'Vocab size':<25} {vocab_size:>15,} {gpt2_vocab:>15,}")
print(f"{'Embed dim':<25} {EMBED_DIM:>15,} {gpt2_dim:>15,}")
print(f"{'Max positions':<25} {MAX_SEQ_LEN:>15,} {gpt2_max_pos:>15,}")
print("-" * 57)
print(f"{'Token emb params':<25} {tok_params:>15,} {gpt2_tok_params:>15,}")
print(f"{'Position emb params':<25} {pos_params:>15,} {gpt2_pos_params:>15,}")
print(f"{'Total embedding params':<25} {total_emb_params:>15,} {gpt2_total_emb:>15,}")
print(f"\nGPT-2 Small has {gpt2_total_emb / total_emb_params:.0f}x more embedding parameters than our toy model.")

---
## Key Insights

1. **BPE is a simple, greedy algorithm.** Start with bytes, count adjacent pair frequencies, merge the most frequent pair, repeat. With enough merges on enough data, it discovers meaningful subword units (morphemes, common words, punctuation patterns).

2. **BPE operates at the byte level in GPT-2.** Because every possible byte is in the base vocabulary, the tokenizer can handle *any* input -- no unknown tokens, ever. This is a major practical advantage over word-level or character-level approaches.

3. **Token embeddings are a learned lookup table.** `nn.Embedding(vocab_size, embed_dim)` is just a matrix. Passing token ID `i` returns row `i`. Gradients during training only update the rows that were actually used in each batch.

4. **GPT-2 uses learned (not sinusoidal) positional embeddings.** Another `nn.Embedding(max_seq_len, embed_dim)`, indexed by position. The model learns what each absolute position "means." This differs from the original Transformer's fixed sinusoidal scheme.

5. **The combined embedding is a simple sum.** `output = token_emb(token_ids) + pos_emb(positions)`. This is the input to the first transformer block. The sum lets the model learn to encode both identity and position in the same vector space.

6. **Without positional embeddings, order is invisible.** The same tokens in any arrangement produce the same bag of vectors. Positional embeddings break this symmetry, giving the attention mechanism the ability to distinguish "A before B" from "B before A."

7. **In a trained model, nearby positions have similar embeddings.** The positional similarity decays smoothly with distance, giving the model a notion of locality. This structure is *learned*, not imposed -- the model discovers it because locality matters for language.